In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
import pycircstat2 as cs2
ss = hf.settings_dict()

In [2]:
baseline_tmin = -0.3
baseline_tmax = 0.0
stimulus_tmin = 0.7
stimulus_tmax = 3.7



In [3]:
from pycircstat2.hypothesis import wheeler_watson_test, watson_u2_test

for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)

        stc_baseline = stc.copy().crop(baseline_tmin, baseline_tmax)
        stc_stimulus = stc.copy().crop(stimulus_tmin, stimulus_tmax)


        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        w_map = np.zeros(n_voxels)
        p_map = np.zeros(n_voxels)

        x = np.asarray(stc_baseline.data[0]).ravel()
        y = np.asarray(stc_stimulus.data[0]).ravel()

        x = (x + 2*np.pi) % (2*np.pi)
        y = (y + 2*np.pi) % (2*np.pi)

        print(len(np.unique(np.concatenate([x, y]))))
        print(len(np.concatenate([x, y])))

        for voxel_idx in range(n_voxels):
            x = np.asarray(stc_baseline.data[voxel_idx]).ravel()
            y = np.asarray(stc_stimulus.data[voxel_idx]).ravel()

            x = x[np.isfinite(x)]
            y = y[np.isfinite(y)]

            # tiny jitter to avoid ranking/tie bugs
            eps = 1e-10
            x = x + np.random.normal(0, eps, size=x.shape)
            y = y + np.random.normal(0, eps, size=y.shape)

            # wrap angles
            x = (x + 2*np.pi) % (2*np.pi)
            y = (y + 2*np.pi) % (2*np.pi)

            result = watson_u2_test([x, y])

            w_map[voxel_idx] = result.U2
            p_map[voxel_idx] = result.p_val


        print(f"delta_r min  : {w_map.min():.4f}")
        print(f"delta_r max  : {w_map.max():.4f}")
        print(f"delta_r mean : {w_map.mean():.4f}")
        print(f"Voxels > 0   : {(w_map > 0).sum()} / {n_voxels}")

        z_stc       = stc.copy().crop(0, 0.0 + 1/ss['fs'])
        z_stc.data  = w_map[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)


loading dataset for subject:  0005_3SJ
6598
6602


ValueError: operands could not be broadcast together with shape (602,) (651,)